In [110]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [111]:
df = pd.read_excel("../data/raw/tashkent_houses.xlsx")
df.head()

,address,district,rooms,size,level,max_levels,price,lat,lng
0,"город Ташкент, Yunusobod район, Юнусабад 8-й к...",Yunusobod,3,57.0,4,4,52000,41.371471,69.281049
1,"город Ташкент, Yakkasaroy район, 1-й тупик Шот...",Yakkasaroy,2,52.0,4,5,56000,41.291115,69.261104
2,"город Ташкент, Chilonzor район, Чиланзар 2-й к...",Chilonzor,2,42.0,4,4,37000,41.280784,69.223683
3,"город Ташкент, Chilonzor район, Чиланзар 9-й к...",Chilonzor,3,65.0,1,4,49500,41.290163,69.196862
4,"город Ташкент, Chilonzor район, площадь Актепа",Chilonzor,3,70.0,3,5,55000,41.300156,69.210831


In [112]:
df = df.drop('address', axis=1)
df.head()

,district,rooms,size,level,max_levels,price,lat,lng
0,Yunusobod,3,57.0,4,4,52000,41.371471,69.281049
1,Yakkasaroy,2,52.0,4,5,56000,41.291115,69.261104
2,Chilonzor,2,42.0,4,4,37000,41.280784,69.223683
3,Chilonzor,3,65.0,1,4,49500,41.290163,69.196862
4,Chilonzor,3,70.0,3,5,55000,41.300156,69.210831


In [113]:
# Datasetni qisqartramiz price 400 000 size esa 350 kv.m ga qilib olamiz bizga bu bashoratni aniqroq qiladi.
df = df[(df['price'] <= 400000) & (df['size'] <= 350)]
df.shape

(7405, 8)

In [114]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42,stratify=df['district'])
print(f"Train set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")


Train set shape: (5924, 8)
Test set shape: (1481, 8)


In [115]:
#If there are missing values in a dataset, we can use SimpleImputer to fill them with the mean of each column.
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
imputer

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [116]:
from sklearn.preprocessing import OrdinalEncoder
encoder_ordinal = OrdinalEncoder()

houses_ds_encoded = encoder_ordinal.fit_transform(df_train[['district']])
houses_ds_encoded[:10]

array([[ 3.],
       [ 7.],
       [11.],
       [ 2.],
       [ 3.],
       [10.],
       [10.],
       [ 1.],
       [ 6.],
       [ 3.]])

In [117]:
from sklearn.preprocessing import OneHotEncoder
encoder_onehot = OneHotEncoder()

houses_ds_encoded = encoder_onehot.fit(df_train[['district']])
houses_ds_encoded = encoder_onehot.transform(df_test[['district']])
column_names = encoder_onehot.get_feature_names_out(['district'])


district = pd.DataFrame(houses_ds_encoded.toarray(), columns=column_names)
district.columns = district.columns.str.replace('district_', '')
district.head()


,Bektemir,Chilonzor,Mirobod,Mirzo Ulugbek,Olmzor,Sergeli,Shayhontohur,Uchtepa,Yakkasaroy,Yangihayot,Yashnobod,Yunusobod
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [118]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, size_ix, lat_ix, lng_ix = 0, 1, 5, 6

class CustomFeaturesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, center_lat=41.3171, center_lng=69.2797):
        # Markaz kordinatalarini initsializatsiya qilamiz
        self.center_lat = center_lat
        self.center_lng = center_lng
        
    def fit(self, X, y=None):
        return self # Transformer faqat o'zgartiradi, hech narsa o'rganmaydi
        
    def transform(self, X):
        # 1. room_size hisoblash (size / rooms)
        room_size = X[:, size_ix] / X[:, rooms_ix]
        
        # 2. distance_to_center hisoblash (Haverzin)
        R = 6371 # Yer radiusi
        
        # X massividan lat va lng ustunlarini olib, radianga o'giramiz
        lat1, lon1 = map(np.radians, [X[:, lat_ix], X[:, lng_ix]])
        lat2, lon2 = map(np.radians, [self.center_lat, self.center_lng])
        
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        distance_to_center = R * c
        
        # Asl X massiviga yangi 2 ta ustunni qo'shib qaytaramiz (np.c_ orqali)
        return np.c_[X, room_size, distance_to_center]

In [119]:
adder = CustomFeaturesAdder()
df_train_num = df_train.drop('district', axis=1)
X_train_prepared = adder.transform(df_train_num.values)

print(X_train_prepared[:1])

[[4.00000000e+00 1.20000000e+02 4.00000000e+00 8.00000000e+00
  1.60000000e+05 4.13312004e+01 6.93092476e+01 3.00000000e+01
  2.92341134e+00]]


In [123]:
df_train_num

,rooms,size,level,max_levels,price,lat,lng
3441,4,120.00,4,8,160000,41.331200,69.309248
6075,3,70.00,5,5,44000,41.281422,69.176001
2880,4,82.00,3,4,81000,41.329887,69.280304
1891,4,120.00,1,4,135000,41.291072,69.276913
4081,4,92.00,4,5,54600,41.332655,69.368954
...,...,...,...,...,...,...,...
2089,1,38.51,3,4,21000,41.361970,69.387577
1706,3,90.00,3,5,102000,41.291072,69.276913
7225,1,45.00,3,5,29500,41.349320,69.386068
2382,2,60.00,4,8,55000,41.314638,69.322862


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_prepared)
X_train_scaled[:1]

array([[0.33333333, 0.31547619, 0.17647059, 0.29166667, 0.39707835,
        0.63239302, 0.41844039, 0.18518519, 0.12807003]])

In [166]:
from sklearn.preprocessing import StandardScaler
scaler_std = StandardScaler()

X_train_scaled_std = scaler_std.fit(X_train_prepared)
X_train_scaled_std = scaler_std.transform(X_train_prepared)

X_train_scaled_std[:1]

array([[ 1.30661963,  1.35526899,  0.1400949 ,  0.7584311 ,  2.3992427 ,
         0.81949687,  0.88977987,  0.18023521, -1.00570271]])

In [168]:
df.to_csv("../data/processed/tashkent_houses.csv", index=False)